Read the API Key

In [ ]:
from src.data_loader import load_all_documents 
from src.embedding import EmbeddingPipeline
from src.vectorstore import FaissVectorStore
from langchain_community.vectorstores import FAISS
from src.search import RAGSearch
import os
from dotenv import load_dotenv
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

d:\Master_RAG\.DGA_Env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from src.logger import prepare_logging
prepare_logging()

In [ ]:
#1. Read the data 
docs = load_all_documents(data_dir="./data")

#2. Chunking/Embeddinbg the data 
embedding = EmbeddingPipeline()
chunks, embeded_data = embedding.chunk_documents(docs)


#3. create vector database
store = FaissVectorStore("faiss_store")
store.build_from_documents(embeded_data, chunks)

#4. 
#retriever = store.as_retriever(search_kwargs={"k": 3})
RAGResume = RAGSearch(groq_api_key=groq_api_key, embeddings=embeded_data, chunks=chunks)
query = "Summarize this resume in bullet points?"
result = RAGResume.search_and_summarize(query, top_k=3)
print(f"--- RAG Response ---\n{result}")


2026-03-22 07:31:31,174 - [DEBUG] Data path: D:\Master_RAG\data
2026-03-22 07:31:31,181 - [DEBUG] Found 0 PDF files: []
2026-03-22 07:31:31,182 - [DEBUG] Found 0 TXT files: []
2026-03-22 07:31:31,183 - [DEBUG] Found 0 CSV files: []
2026-03-22 07:31:31,184 - [DEBUG] Found 0 Excel files: []
2026-03-22 07:31:31,185 - [DEBUG] Found 1 Word files: ['D:\\Master_RAG\\data\\Data Scientist updated.docx']
2026-03-22 07:31:31,185 - [DEBUG] Loading Word: D:\Master_RAG\data\Data Scientist updated.docx
2026-03-22 07:31:31,195 - [DEBUG] Loaded 1 Word docs from D:\Master_RAG\data\Data Scientist updated.docx
2026-03-22 07:31:31,198 - [DEBUG] Found 0 JSON files: []
2026-03-22 07:31:31,198 - [DEBUG] Total loaded documents: 1
d:\Master_RAG\src\embedding.py:13: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip i

--- RAG Response ---
Here's a summarized version of the resume in bullet points:

**Key Highlights:**

* Designed a KPI-driven tracking system to automate project reporting and monitor timelines, metrics, and outcomes.
* Automated the HR application process using Google Forms, increasing hiring decision speed by 10x while ensuring compliance with data privacy regulations.
* Improved client satisfaction by delivering custom reports and dashboards aligned with business needs.
* Collaborated on ERP expansion, gathering requirements, and designing UML and ERD diagrams for new modules.
* Applied data analytics to evaluate the company's bonus program, leading to a redesigned system that boosted employee satisfaction by 34%.
* Authored user guides, technical documentation, and training materials to enhance client IT teams' incident resolution abilities.

**Professional Experience:**

* **SQL Engineer (YemenSoft Inc., Yemen, 2016-2018)**
 + Improved client satisfaction through custom reports a

In [8]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from datasets import Dataset
import pandas as pd

# --- Step 5: Prepare Evaluation Data ---

# We need the actual retrieved chunks to check if the LLM hallucinated
retrieved_docs = store.search(query, k=3) 
context_content = [doc.page_content for doc in retrieved_docs]

# Create the dataset for Ragas
data = {
    "question": [query],
    "answer": [result],
    "contexts": [context_content],
}
dataset = Dataset.from_dict(data)

# --- Step 6: Run Evaluation (Using Groq as the Judge) ---

# Ragas needs to know which LLM to use to "grade" the answers
# You can reuse your RAGResume.llm here
result_score = evaluate(
    dataset,
    metrics=[faithfulness, answer_relevancy], #context_precision for supervised RAG
    llm=RAGResume.llm, 
    embeddings=embedding.embeddings # Your HuggingFace model
)

# --- Step 7: View Results ---
df = result_score.to_pandas()
print("\n--- Evaluation Metrics ---")
print(df[['faithfulness', 'answer_relevancy']])


C:\Users\engam\AppData\Local\Temp\ipykernel_5556\3608784270.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy
C:\Users\engam\AppData\Local\Temp\ipykernel_5556\3608784270.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy



--- Result 1 ---
Content: Designed a KPI-driven tracking system to automate project reporting and monitor timelines, metrics, and outcomes.

Automated the HR application process with Google Forms, enabling management to make h...
Metadata: {'source': 'D:\\Master_RAG\\data\\Data Scientist updated.docx'}

--- Result 2 ---
Content: Authored user guides, technical documentation, and training materials, enhancing client IT teams’ ability to resolve incidents independently



Academics       

Master’s degree in Computer Science - ...
Metadata: {'source': 'D:\\Master_RAG\\data\\Data Scientist updated.docx'}

--- Result 3 ---
Content: Github: https://github.com/amrsaeed0092?tab=repositories

LinkedIn: https://www.linkedin.com/in/amr-saeed-86a558223





Professional Experience 

Lead Analytics Consultant

Wells Fargo, Dallas, Texas...
Metadata: {'source': 'D:\\Master_RAG\\data\\Data Scientist updated.docx'}


Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]2026-03-22 07:43:05,823 - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 400 Bad Request"
2026-03-22 07:43:05,829 - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 400 Bad Request"
2026-03-22 07:43:06,519 - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 400 Bad Request"
2026-03-22 07:43:07,035 - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-22 07:43:07,256 - LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
2026-03-22 07:43:08,001 - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
Evaluating: 100%|██████████| 2/2 [00:04<00:00,  2.42s/it]



--- Evaluation Metrics ---
   faithfulness  answer_relevancy
0           1.0          0.710336
